# Clinical NLP and LLM Pipeline for Hospital Discharge Summaries

## Objective
The goal of this project is to convert unstructured hospital discharge summaries (PDF documents) into a structured dataset using Natural Language Processing techniques and Large Language models.
PDF → Text → Regex/NLP extraction → LLM JSON extraction → Structured dataset

## Dataset Source

This project uses a preview subset of the **Healthcare Discharge Summary Dataset** available on Kaggle.

The dataset contains a small sample of hospital discharge summary reports provided for data inspection and experimentation.

Each report includes clinical information such as:

- Patient demographics
- Final diagnosis
- Symptoms
- Procedures
- Medications

These PDF reports are used to demonstrate how Natural Language Processing can extract structured clinical information from unstructured medical documents.

Dataset Link:  
https://www.kaggle.com/datasets/infobayai/healthcare-discharge-summary

## Data Access

The dataset is **not included in this repository** due to dataset licensing restrictions.

To reproduce this project:

1. Download the dataset from Kaggle.
2. Place the PDF files inside the `data/` folder.
3. Run the Jupyter Notebook to execute the NLP pipeline.

## Step 1: Import Required Libraries

In [325]:
import pdfplumber
import pandas as pd
import spacy
import os
import re
import openai

In [265]:
nlp = spacy.load("en_core_web_sm")

## Step 2: Define Function to Extract Text from PDF

In [267]:
def extract_text_from_pdf(filepath):
    text =""
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text
text = extract_text_from_pdf("../data/Discharge Summary Sample/PA01.pdf")
print(text[:500])

Discharge Summary
Patient Name : PA01 Age / Sex : 10 Y/ Male
Address : IPD No. :
UHID No. : Discharge Date : 11/04/2024 20:14
Mob No. : Ward :
Admission Date : 09/04/2024 20:08 Category :
Room No. :
Department : General Surgery
Doctor Incharge :
ICD CODE :
.
(L02.31)
FINAL DIAGNOSIS :
RIGHT GLUTEAL ABSCESS
CHIEF COMPLAINTS & REASON FOR ADMISIION :
C\O- FEVER WITH H\O VOMITING
- PAIN IN RIGHT HYPOCHONDRIUM REGION
-BURNING MICTURITION
-PAIN WITH SWELLING RIGHT GLUTEAL REGION
-
HISTORY :
10 YEARS O


## Step 3: Load All Discharge Summary PDFs

In [269]:
filepath = "../data/Discharge Summary Sample"
records = []
for file in os.listdir(filepath):
    if file.endswith(".pdf"):
        text = extract_text_from_pdf(os.path.join(filepath,file))
        records.append({
            "file"  : file,
             "text" : text})
        

In [270]:
df = pd.DataFrame(records)
df.head()


,file,text
0,PA01.pdf,Discharge Summary\nPatient Name : PA01 Age / S...
1,PA02.pdf,Discharge Summary\nPatient Name : PA02 Age / S...
2,PA03.pdf,Discharge Summary\nPatient Name : PA03 Age / S...
3,PA04.pdf,Discharge Summary\nPatient Name : PA04 Age / S...
4,PA05.pdf,Discharge Summary\nPatient Name :PA05 Age / Se...


## Step 4: Verify Extracted Text from PDFs

In [272]:
df.iloc[0]

file                                             PA01.pdf
text    Discharge Summary\nPatient Name : PA01 Age / S...
Name: 0, dtype: object

In [273]:
print(df.iloc[0]["text"][:500])

Discharge Summary
Patient Name : PA01 Age / Sex : 10 Y/ Male
Address : IPD No. :
UHID No. : Discharge Date : 11/04/2024 20:14
Mob No. : Ward :
Admission Date : 09/04/2024 20:08 Category :
Room No. :
Department : General Surgery
Doctor Incharge :
ICD CODE :
.
(L02.31)
FINAL DIAGNOSIS :
RIGHT GLUTEAL ABSCESS
CHIEF COMPLAINTS & REASON FOR ADMISIION :
C\O- FEVER WITH H\O VOMITING
- PAIN IN RIGHT HYPOCHONDRIUM REGION
-BURNING MICTURITION
-PAIN WITH SWELLING RIGHT GLUTEAL REGION
-
HISTORY :
10 YEARS O


In [274]:
df["file"]

0    PA01.pdf
1    PA02.pdf
2    PA03.pdf
3    PA04.pdf
4    PA05.pdf
5    PA06.pdf
6    PA07.pdf
7    PA08.pdf
8    PA09.pdf
9    PA10.pdf
Name: file, dtype: object

In [275]:
len(df)

10

## Step 5: Extract Patient Age and Gender

In [277]:
def extract_age_gender(text):
    pattern = r"Age\s*/\s*Sex\s*:\s*(\d+)\s*Y\s*/\s*([A-Za-z]+)"
    match = re.search(pattern, text, re.IGNORECASE)

    if match:
        age = match.group(1)
        gender = match.group(2)
        return age, gender

    return None, None

In [278]:
df[["age","gender"]] = df["text"].apply(lambda x: pd.Series(extract_age_gender(x)))

In [279]:
df[["file","age","gender"]]

,file,age,gender
0,PA01.pdf,10,Male
1,PA02.pdf,52,Female
2,PA03.pdf,61,Female
3,PA04.pdf,60,Male
4,PA05.pdf,19,Male
5,PA06.pdf,32,Female
6,PA07.pdf,26,Female
7,PA08.pdf,70,Female
8,PA09.pdf,22,Male
9,PA10.pdf,66,Female


## Step 6: Extract Final Diagnosis

In [281]:
def extract_diagnosis(text):
    pattern = r"FINAL DIAGNOSIS\s*:\s*(.*)"
    match = re.search(pattern, text)
    if match:
        return match.group(1).strip()
    return None

df["diagnosis"] = df["text"].apply(extract_diagnosis)
df[["file","diagnosis"]]

,file,diagnosis
0,PA01.pdf,RIGHT GLUTEAL ABSCESS
1,PA02.pdf,DENGUE FEVER
2,PA03.pdf,ACCELERATED HYPERTENSION
3,PA04.pdf,BPH /UTI
4,PA05.pdf,DENGUE IGG WEAKLY +
5,PA06.pdf,G3P2 L2 WITH POG 37 WEEKS IN SPONTANEOUS LABOR.
6,PA07.pdf,G2P1L1/36+3WEEKS/PREV. LSCS/SHORT ICP/RH NEGATIVE
7,PA08.pdf,RIGHT MCA INFARCT WITH LEFT HEMIPARESIS
8,PA09.pdf,RIGHT PNEUMOTHORAX
9,PA10.pdf,ACUTE ON CKD


In [282]:
df["diagnosis"] = df["diagnosis"].str.title()

In [283]:
df["diagnosis"] = df["diagnosis"].str.replace("/", " / ")
df["diagnosis"] = df["diagnosis"].str.replace("+", " + ")
df["diagnosis"] = df["diagnosis"].str.replace("  ", " ")

In [284]:
df[["file","diagnosis"]]

,file,diagnosis
0,PA01.pdf,Right Gluteal Abscess
1,PA02.pdf,Dengue Fever
2,PA03.pdf,Accelerated Hypertension
3,PA04.pdf,Bph / Uti
4,PA05.pdf,Dengue Igg Weakly +
5,PA06.pdf,G3P2 L2 With Pog 37 Weeks In Spontaneous Labor.
6,PA07.pdf,G2P1L1 / 36 + 3Weeks / Prev. Lscs / Short Icp ...
7,PA08.pdf,Right Mca Infarct With Left Hemiparesis
8,PA09.pdf,Right Pneumothorax
9,PA10.pdf,Acute On Ckd


## Step 7: Extract Symptoms

In [286]:
symptoms_keywords = [
    "fever","vomiting","pain","weakness",
    "cough","burning micturition","nausea",
    "bodyache","chills","hematuria","slurred speech"
]

def extract_symptoms(text):
    found = []
    lower_text = text.lower()

    for symptom in symptoms_keywords:
        if symptom in lower_text:
            found.append(symptom)

    return found

In [287]:
df["symptoms"] = df["text"].apply(extract_symptoms)

In [288]:
df["symptoms"] = df["symptoms"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

In [289]:
df[["file","symptoms"]]

,file,symptoms
0,PA01.pdf,"fever, vomiting, pain, burning micturition"
1,PA02.pdf,"fever, weakness, nausea, bodyache"
2,PA03.pdf,"pain, nausea"
3,PA04.pdf,"fever, chills, hematuria"
4,PA05.pdf,"fever, vomiting, pain, weakness, nausea, chills"
5,PA06.pdf,fever
6,PA07.pdf,"pain, cough"
7,PA08.pdf,"weakness, slurred speech"
8,PA09.pdf,"pain, cough"
9,PA10.pdf,"fever, vomiting, pain"


## Step 8: Extract Medical Procedures

In [291]:
def extract_procedures(text):
    pattern = r"SURGERY DETAILS\s*:\s*(.*?)(?:CONDITION AT DISCHARGE|ADVISE ON DISCHARGE)"
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)

    if match:
        return " ".join(match.group(1).split())

    return None

In [292]:
df["procedures"] = df["text"].apply(extract_procedures)

In [293]:
df[["file","procedures"]]

,file,procedures
0,PA01.pdf,DIAGNOSIS - RIGHT GLUTEAL ABSCESS PROCEDURE - ...
1,PA02.pdf,NA Discharge Summary
2,PA03.pdf,None
3,PA04.pdf,NA
4,PA05.pdf,None
5,PA06.pdf,PROCEDURE- NVD WITH RMLE DONE ON 07/10/25. STE...
6,PA07.pdf,SURGERY- ELECTIVE LSCS ANAESTHESIA- S.A POSITI...
7,PA08.pdf,NA
8,PA09.pdf,None
9,PA10.pdf,NA


In [294]:
df["procedures"] = df["procedures"].str.replace("PROCEDURE-", "", regex=False)
df["procedures"] = df["procedures"].str.strip()

In [295]:
df[["file","procedures"]]

,file,procedures
0,PA01.pdf,DIAGNOSIS - RIGHT GLUTEAL ABSCESS PROCEDURE - ...
1,PA02.pdf,NA Discharge Summary
2,PA03.pdf,None
3,PA04.pdf,NA
4,PA05.pdf,None
5,PA06.pdf,NVD WITH RMLE DONE ON 07/10/25. STEPS- PATIENT...
6,PA07.pdf,SURGERY- ELECTIVE LSCS ANAESTHESIA- S.A POSITI...
7,PA08.pdf,NA
8,PA09.pdf,None
9,PA10.pdf,NA


## Step 9: Extract Medications

In [297]:
def extract_medications(text):

    pattern = r"ADVICE ON DISCHARGE MEDICINE(.*)"
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)

    if not match:
        return []

    med_text = match.group(1)
    lines = med_text.split("\n")

    meds = []

    for line in lines:
        line = line.strip()
        if line and len(line) < 40:
            meds.append(line)

    return meds[:10]

In [298]:
df["medications"] = df["text"].apply(extract_medications)

In [299]:
df["medications"] = df["medications"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

In [300]:
df[["file","medications"]]

,file,medications
0,PA01.pdf,TWICE A DAY
1,PA02.pdf,"1 PCM No No No No 0, 2 TAB BOOSTBOON No No No ..."
2,PA03.pdf,
3,PA04.pdf,"1 TAB PANTOP 0 No No No No 0, 2 SYP ALKACIP No..."
4,PA05.pdf,
5,PA06.pdf,"3 TAB DOLO TWICE A DAY Yes No Yes No 3, 4 CAP...."
6,PA07.pdf,
7,PA08.pdf,
8,PA09.pdf,
9,PA10.pdf,"4 TAB. PAN 40MG MEA Yes No No No 5, ONCE A DAY..."


## Step 10: Create Structured Clinical Dataset

In [310]:
import numpy as np

def clean_value(x):
    if pd.isna(x):
        return "Not available"
    
    x_str = str(x).strip()
    
    if x_str == "" or x_str.lower() in ["na", "n/a", "na discharge summary"]:
        return "Not available"
    
    return x_str

In [312]:
cols = ["age","gender","diagnosis","symptoms","procedures","medications"]

for col in cols:
    df[col] = df[col].apply(clean_value)

In [314]:
df = df.fillna("Not Available")

In [316]:
df[cols]

,age,gender,diagnosis,symptoms,procedures,medications
0,10,Male,Right Gluteal Abscess,"fever, vomiting, pain, burning micturition",DIAGNOSIS - RIGHT GLUTEAL ABSCESS PROCEDURE - ...,TWICE A DAY
1,52,Female,Dengue Fever,"fever, weakness, nausea, bodyache",Not available,"1 PCM No No No No 0, 2 TAB BOOSTBOON No No No ..."
2,61,Female,Accelerated Hypertension,"pain, nausea",Not Available,Not available
3,60,Male,Bph / Uti,"fever, chills, hematuria",Not available,"1 TAB PANTOP 0 No No No No 0, 2 SYP ALKACIP No..."
4,19,Male,Dengue Igg Weakly +,"fever, vomiting, pain, weakness, nausea, chills",Not Available,Not available
5,32,Female,G3P2 L2 With Pog 37 Weeks In Spontaneous Labor.,fever,NVD WITH RMLE DONE ON 07/10/25. STEPS- PATIENT...,"3 TAB DOLO TWICE A DAY Yes No Yes No 3, 4 CAP...."
6,26,Female,G2P1L1 / 36 + 3Weeks / Prev. Lscs / Short Icp ...,"pain, cough",SURGERY- ELECTIVE LSCS ANAESTHESIA- S.A POSITI...,Not available
7,70,Female,Right Mca Infarct With Left Hemiparesis,"weakness, slurred speech",Not available,Not available
8,22,Male,Right Pneumothorax,"pain, cough",Not Available,Not available
9,66,Female,Acute On Ckd,"fever, vomiting, pain",Not available,"4 TAB. PAN 40MG MEA Yes No No No 5, ONCE A DAY..."


## Step 11: Save Structured Dataset

In [322]:
df.to_csv("structured_discharge_dataset.csv", index=False)

## Step 12: LLM-Based Structured Clinical Information Extraction

In [329]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

In [337]:
load_dotenv()
print(os.getenv("OPENAI_API_KEY")[:10])

sk-proj-RZ


In [339]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [361]:
def extract_all_files_llm(records):
    prompt = f"""
You are a clinical information extraction assistant.

You will receive multiple hospital discharge summaries.
For each record, extract structured clinical information.

Return ONLY valid JSON.
Return a JSON array, where each item has this schema:

{{
  "file": "string",
  "patient_age": "string",
  "gender": "string",
  "diagnosis": "string",
  "symptoms": ["list of symptoms"],
  "medications": ["list of medications"],
  "procedures": ["list of procedures"]
}}

Rules:
- Preserve the file name exactly.
- If a field is missing, return "Not available".
- symptoms, medications, and procedures must always be arrays.
- Do not include explanations, comments, or markdown fences.
- Return JSON only.

Input records:
{json.dumps(records, indent=2)}
"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {
                "role": "system",
                "content": "You extract structured clinical data from discharge summaries and return only valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
    
    )

    content = response.choices[0].message.content.strip()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        print("JSON parsing failed. Raw response below:")
        print(content)
        return None

In [351]:
records_for_llm = []

for _, row in df.iterrows():
    
    record = {
        "file": row["file"],
        "text": row["text"][:2500]   # limit text length
    }
    
    records_for_llm.append(record)

In [353]:
len(records_for_llm)

10

In [357]:
records_for_llm[0]

{'file': 'PA01.pdf',
 'text': 'Discharge Summary\nPatient Name : PA01 Age / Sex : 10 Y/ Male\nAddress : IPD No. :\nUHID No. : Discharge Date : 11/04/2024 20:14\nMob No. : Ward :\nAdmission Date : 09/04/2024 20:08 Category :\nRoom No. :\nDepartment : General Surgery\nDoctor Incharge :\nICD CODE :\n.\n(L02.31)\nFINAL DIAGNOSIS :\nRIGHT GLUTEAL ABSCESS\nCHIEF COMPLAINTS & REASON FOR ADMISIION :\nC\\O- FEVER WITH H\\O VOMITING\n- PAIN IN RIGHT HYPOCHONDRIUM REGION\n-BURNING MICTURITION\n-PAIN WITH SWELLING RIGHT GLUTEAL REGION\n-\nHISTORY :\n10 YEARS OLD MALE PATIENT PRESENTED IN OUR HOSPITAL WITH C\\O FEVER WITH H\\O VOMITING\nSINCE 7-8 DAYS WITH BURNING MICTURITION WITH PAIN RIGHT HYPOCHONDRIUM WITH PAIN WITH\nSWELLING RIGHT GLUTEAL REGION .THEN PATIENT GOT ADMITTED IN OUR HOSPITAL FOR FURTHER\nINVESTIGATIONS AND MANAGEMENT .\nN\\H\\O ANY PAST ILLNESS\nLOCAL EXAMINATION :\nCVS-S1 S2 +\nPR: 82/MIN, BP: 100\\60MMHG, RR: 22/MIN, TEMP: 98.6°F-\nCHEST -B\\L A\\E +\nCNS - CONSCIOUS\\ORIENTED .

In [363]:
llm_results = extract_all_files_llm(records_for_llm)

In [369]:
llm_results[2]

{'file': 'PA03.pdf',
 'patient_age': '61 Y',
 'gender': 'Female',
 'diagnosis': 'ACCELERATED HYPERTENSION',
 'symptoms': ['Nausea',
  'Anxiety',
  'Pain in epigastric region',
  'High blood pressure',
  'Headache'],
 'medications': ['Nitroglycerin infusion',
  'Antihypertensive drugs',
  'IV fluids',
  'Multivitamins',
  'PPI',
  'Antiemetics',
  'Etoshine 90 mg',
  'Shelcal Joints',
  'Bio D3 Fem',
  'Ciplox D ear drops',
  'Cefuroxime',
  'Abroxol 500 mg',
  'Calpol (paracetamol)',
  'Supradyn'],
 'procedures': ['MRI brain',
  'Nitroglycerin infusion (as antihypertensive measure)']}

In [371]:
df_llm = pd.DataFrame(llm_results)
df_llm.head()

,file,patient_age,gender,diagnosis,symptoms,medications,procedures
0,PA01.pdf,10 Y,Male,RIGHT GLUTEAL ABSCESS,"[Fever, Vomiting, Pain in right hypochondrium,...","[IV fluids, IV antibiotics, PPI, Analgesics]","[Incision and drainage, Spinal anesthesia]"
1,PA02.pdf,52 Y,Female,DENGUE FEVER,"[Fever, Bodyache, Nausea, General weakness, De...","[IV fluids, IV medications]",[Not available]
2,PA03.pdf,61 Y,Female,ACCELERATED HYPERTENSION,"[Nausea, Anxiety, Pain in epigastric region, H...","[Nitroglycerin infusion, Antihypertensive drug...","[MRI brain, Nitroglycerin infusion (as antihyp..."
3,PA04.pdf,60 Y,Male,BPH /UTI,"[Fever with chills, Urinary incontinence, Mild...",[IV antibiotics],[Catheterisation]
4,PA05.pdf,19 Y,Male,DENGUE IGG WEAKLY +,"[Fever with chills, Generalized weakness]","[IV antibiotics, IV fluids, Antacids, TAB CEFT...",[Not available]


In [373]:
df_final = df.merge(df_llm, on="file", how="left")

In [377]:
df_final.to_csv("../results/structured_discharge_dataset_llm.csv", index=False)

## Conclusion

In this project, an end-to-end clinical NLP pipeline was developed to convert unstructured hospital discharge summaries into a structured dataset. The workflow began with extracting raw text from PDF documents using `pdfplumber`, followed by preprocessing and rule-based information extraction using regular expressions.

Key clinical attributes such as **patient age, gender, diagnosis, symptoms, medications, and procedures** were successfully extracted and organized into a structured dataframe suitable for downstream analytics and visualization.

To improve extraction robustness, an additional **LLM-based information extraction layer** was implemented. Compared to rule-based methods, the LLM approach demonstrated greater flexibility and accuracy when handling variations in clinical language and document formatting.

The final structured dataset enables further healthcare analytics and visualization, including dashboards built with tools such as **Power BI**.

This project demonstrates how **Natural Language Processing and Large Language Models can transform unstructured clinical documents into structured, machine-readable data**, supporting scalable healthcare data analysis and decision-making.